# SVM geometry: signed scores, margins, and one kernel

Companion to HTB Academy Module 290, Section 8. This notebook **checks** a hand-chosen hard-margin separator and illustrates the polynomial kernel trick. It does not fit an SVM or claim a real security classifier.

Read $w\cdot x+b=0$ as *w dot x plus b equals zero*: points satisfying it lie on the decision boundary. Read $y_i(w\cdot x_i+b)\ge1$ as *y sub i times w dot x sub i plus b is at least one*. That is HTB's hard-margin constraint for each training row.

In [ ]:
from math import sqrt

# A tiny 2D feature space; only the first coordinate affects this boundary.
weights = (1.0, 0.0)  # w
bias = -1.0         # b, so the decision boundary is x[0] = 1.
rows = [((0.0, 0.0), -1), ((2.0, 0.0), +1)]  # (x_i, y_i)

def dot(left, right):
    # Dot product: sum_j left_j * right_j.
    return sum(a * b for a, b in zip(left, right))

weight_length = sqrt(dot(weights, weights))  # ||w||, Euclidean length.
for features, label in rows:
    score = dot(weights, features) + bias  # w dot x_i + b.
    signed_score = label * score          # y_i(w dot x_i + b).
    distance = abs(score) / weight_length # Perpendicular boundary distance.
    print(f'features={features}, label={label:+d}, score={score:+.1f}, '
          f'signed score={signed_score:.1f}, distance={distance:.1f}')
    assert signed_score >= 1  # Both satisfy HTB's hard-margin condition.
    assert signed_score == 1  # Both exactly touch a margin: support vectors.

objective = 0.5 * dot(weights, weights)  # One-half ||w|| squared.
full_margin_width = 2 / weight_length
print(f'Hard-margin objective: {objective:.1f}; full gap: {full_margin_width:.1f}')
assert objective == 0.5 and full_margin_width == 2.0

## One point on either side is not a calibrated probability

The *sign* of $w\cdot x+b$ determines the side; its magnitude is a score tied to this model's scaling. It is not a posterior probability. The hard-margin constraints require separable data; real data may need soft-margin violations.

In [ ]:
candidate = (1.5, 0.0)
candidate_score = dot(weights, candidate) + bias
print(f'Candidate score: {candidate_score:+.1f}; side: '
      f'{"positive" if candidate_score > 0 else "negative" if candidate_score < 0 else "boundary"}')
assert candidate_score == 0.5

## A polynomial kernel without constructing features in the actual calculation

Read $K(x,z)=(x\cdot z)^2$ as *K of x and z equals x dot z, squared*. For two-dimensional inputs, this equals the dot product of the explicit transformed vectors $\phi(x)=(x_1^2,\sqrt2x_1x_2,x_2^2)$ and $\phi(z)$—read *phi of x equals x sub one squared, square root of two times x sub one times x sub two, x sub two squared*. We compute both sides only to **verify the identity**. A kernel-based SVM can use the left-hand calculation without materializing these three coordinates. This does not train an SVM or guarantee nonlinear separation.

In [ ]:
x, z = (1.0, 2.0), (3.0, 4.0)
kernel_value = dot(x, z) ** 2  # K(x,z): square the original-space dot product.

def explicit_squared_features(point):
    first, second = point
    return (first ** 2, sqrt(2) * first * second, second ** 2)

mapped_dot = dot(explicit_squared_features(x), explicit_squared_features(z))
print(f'Kernel value: {kernel_value:.0f}; mapped-space dot product: {mapped_dot:.0f}')
assert kernel_value == 121
assert abs(kernel_value - mapped_dot) < 1e-12